# Solucionario — Laboratorio 06 — Evaluación y Despliegue de Modelos
### Tecnologías Emergentes (ISO46B) — Semana 6 (material docente)

Versión con el código completo y celdas de verificación automática (✅). Las verificaciones numéricas de desempeño del modelo (accuracy, F1) son **flexibles por diseño**: el dataset se descarga en tiempo real y varía según la fecha de ejecución, así que se valida la estructura y coherencia del flujo, no un valor fijo.


## Parte 1 — Básico: dataset, entrenamiento y evaluación

In [ ]:
!pip install -q flask pytest

In [ ]:
import datetime as dt
import requests
import pandas as pd
import numpy as np

LATITUD, LONGITUD = -12.0654, -75.2049  # Huancayo

fin = dt.date.today() - dt.timedelta(days=5)
inicio = fin - dt.timedelta(days=400)

url = "https://archive-api.open-meteo.com/v1/archive"
parametros = {
    "latitude": LATITUD,
    "longitude": LONGITUD,
    "start_date": inicio.isoformat(),
    "end_date": fin.isoformat(),
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
    "timezone": "America/Lima",
}

respuesta = requests.get(url, params=parametros, timeout=30)
respuesta.raise_for_status()
datos = respuesta.json()["daily"]

df = pd.DataFrame(datos)
df = df.rename(columns={
    "time": "fecha",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "precipitacion",
    "windspeed_10m_max": "viento_max",
})
df = df.dropna().reset_index(drop=True)
print(f"Dataset descargado: {len(df)} filas")
df.head()

In [ ]:
df["lluvia_mañana"] = (df["precipitacion"].shift(-1) > 0).astype(int)
df = df.dropna().reset_index(drop=True)
df = df.drop(columns=["precipitacion"])

COLUMNAS_CARACTERISTICAS = ["temp_max", "temp_min", "viento_max"]
print(df["lluvia_mañana"].value_counts())

# ✅ Verificación automática
assert "lluvia_mañana" in df.columns, "Falta la columna objetivo"
assert set(df["lluvia_mañana"].unique()).issubset({0, 1}), "La columna objetivo debe ser binaria"
assert "precipitacion" not in df.columns, "La columna 'precipitacion' debe eliminarse (evita fuga de información)"
print("✅ Variable objetivo construida correctamente, sin fuga de información.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[COLUMNAS_CARACTERISTICAS].values
y = df["lluvia_mañana"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)

# ✅ Verificación automática
assert abs(len(X_test) / len(X) - 0.2) < 0.02, "La partición de prueba debería ser ~20%"
print("✅ Partición train/test correcta:", X_train_esc.shape, X_test_esc.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers


def construir_modelo(unidades_ocultas=8):
    modelo = keras.Sequential([
        layers.Input(shape=(3,)),
        layers.Dense(unidades_ocultas, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return modelo


modelo = construir_modelo(unidades_ocultas=8)
modelo.fit(X_train_esc, y_train, epochs=60, batch_size=16, validation_split=0.2, verbose=0)

# ✅ Verificación automática
assert modelo.count_params() > 0, "El modelo debe tener parámetros entrenables"
print("✅ Modelo base entrenado. Parámetros:", modelo.count_params())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = (modelo.predict(X_test_esc, verbose=0) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred, digits=3))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicción"); plt.ylabel("Real")
plt.title("Matriz de confusión")
plt.show()

In [ ]:
import json

reporte = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
metricas = {
    "accuracy": reporte["accuracy"],
    "precision_clase_1": reporte["1"]["precision"],
    "recall_clase_1": reporte["1"]["recall"],
    "f1_clase_1": reporte["1"]["f1-score"],
}
with open("metricas_v1.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2)

# ✅ Verificación automática
for clave in ["accuracy", "precision_clase_1", "recall_clase_1", "f1_clase_1"]:
    assert 0.0 <= metricas[clave] <= 1.0, f"{clave} fuera de rango"
print("✅ Métricas calculadas y guardadas:", json.dumps(metricas, indent=2))

## Parte 2 — Intermedio: guardado versionado y comparación

In [ ]:
import pickle

TIMESTAMP = dt.datetime.now().strftime("%Y%m%d_%H%M")
NOMBRE_V1 = f"modelo_lluvia_v1_{TIMESTAMP}.keras"
modelo.save(NOMBRE_V1)
with open("escalador_v1.pkl", "wb") as f:
    pickle.dump(escalador, f)

metadatos_v1 = {
    "version": "v1", "archivo_modelo": NOMBRE_V1, "fecha_entrenamiento": TIMESTAMP,
    "caracteristicas": COLUMNAS_CARACTERISTICAS,
    "hiperparametros": {"unidades_ocultas": 8, "epochs": 60, "batch_size": 16},
    "metricas": metricas,
}
with open("version_info_v1.json", "w", encoding="utf-8") as f:
    json.dump(metadatos_v1, f, indent=2)

import os
assert os.path.exists(NOMBRE_V1), "El archivo del modelo v1 no se guardó"
print(f"✅ Modelo v1 guardado: {NOMBRE_V1}")

In [ ]:
modelo_v2 = construir_modelo(unidades_ocultas=24)
modelo_v2.fit(X_train_esc, y_train, epochs=60, batch_size=16, validation_split=0.2, verbose=0)

y_pred_v2 = (modelo_v2.predict(X_test_esc, verbose=0) > 0.5).astype(int).flatten()
reporte_v2 = classification_report(y_test, y_pred_v2, output_dict=True, zero_division=0)
metricas_v2 = {
    "accuracy": reporte_v2["accuracy"],
    "precision_clase_1": reporte_v2["1"]["precision"],
    "recall_clase_1": reporte_v2["1"]["recall"],
    "f1_clase_1": reporte_v2["1"]["f1-score"],
}

NOMBRE_V2 = f"modelo_lluvia_v2_{TIMESTAMP}.keras"
modelo_v2.save(NOMBRE_V2)
with open("escalador_v2.pkl", "wb") as f:
    pickle.dump(escalador, f)

metadatos_v2 = {
    "version": "v2", "archivo_modelo": NOMBRE_V2, "fecha_entrenamiento": TIMESTAMP,
    "caracteristicas": COLUMNAS_CARACTERISTICAS,
    "hiperparametros": {"unidades_ocultas": 24, "epochs": 60, "batch_size": 16},
    "metricas": metricas_v2,
}
with open("version_info_v2.json", "w", encoding="utf-8") as f:
    json.dump(metadatos_v2, f, indent=2)

print(f"✅ Modelo v2 guardado: {NOMBRE_V2}  (F1={metricas_v2['f1_clase_1']:.3f})")

In [ ]:
with open("version_info_v1.json") as f:
    v1 = json.load(f)
with open("version_info_v2.json") as f:
    v2 = json.load(f)

print(f"v1 -> F1: {v1['metricas']['f1_clase_1']:.3f}")
print(f"v2 -> F1: {v2['metricas']['f1_clase_1']:.3f}")

ganadora = v1 if v1["metricas"]["f1_clase_1"] >= v2["metricas"]["f1_clase_1"] else v2
print(f"Versión promovida a producción: {ganadora['version']}")

import shutil
shutil.copy(ganadora["archivo_modelo"], "modelo_produccion.keras")
shutil.copy(f"escalador_{ganadora['version']}.pkl", "escalador_produccion.pkl")
with open("version_info_produccion.json", "w", encoding="utf-8") as f:
    json.dump(ganadora, f, indent=2)

# ✅ Verificación automática
assert os.path.exists("modelo_produccion.keras"), "No se promovió ningún modelo a producción"
print("✅ Modelo de producción listo.")

## Parte 3 — Avanzado: desplegar como API (probada dentro del notebook)

In [ ]:
%%writefile preprocesamiento_api.py
import numpy as np

COLUMNAS_CARACTERISTICAS = ["temp_max", "temp_min", "viento_max"]


def extraer_caracteristicas(datos_json):
    faltantes = [c for c in COLUMNAS_CARACTERISTICAS if c not in datos_json]
    if faltantes:
        raise ValueError(f"Faltan las siguientes características: {faltantes}")
    valores = [float(datos_json[c]) for c in COLUMNAS_CARACTERISTICAS]
    return np.array([valores])

In [ ]:
import pickle
from flask import Flask, request, jsonify
from preprocesamiento_api import extraer_caracteristicas

app = Flask(__name__)
modelo_api = keras.models.load_model("modelo_produccion.keras")
with open("escalador_produccion.pkl", "rb") as f:
    escalador_api = pickle.load(f)


@app.route("/predecir", methods=["POST"])
def predecir():
    datos_json = request.get_json(silent=True)
    if datos_json is None:
        return jsonify({"error": "Se esperaba un cuerpo JSON válido"}), 400
    try:
        caracteristicas = extraer_caracteristicas(datos_json)
    except ValueError as error:
        return jsonify({"error": str(error)}), 400

    caracteristicas_esc = escalador_api.transform(caracteristicas)
    probabilidad = float(modelo_api.predict(caracteristicas_esc, verbose=0)[0][0])
    prediccion = "lluvia" if probabilidad > 0.5 else "no_lluvia"
    return jsonify({"prediccion": prediccion, "probabilidad": round(probabilidad, 4)})


print("✅ App Flask definida.")

In [ ]:
cliente = app.test_client()

respuesta = cliente.post("/predecir", json={"temp_max": 18.5, "temp_min": 4.2, "viento_max": 12.0})
cuerpo = respuesta.get_json()
print(respuesta.status_code, cuerpo)

# ✅ Verificación automática
assert respuesta.status_code == 200
assert cuerpo["prediccion"] in ("lluvia", "no_lluvia")
assert 0.0 <= cuerpo["probabilidad"] <= 1.0
print("✅ Solicitud válida procesada correctamente.")

In [ ]:
respuesta_error = cliente.post("/predecir", json={"temp_max": 18.5})
cuerpo_error = respuesta_error.get_json()
print(respuesta_error.status_code, cuerpo_error)

# ✅ Verificación automática
assert respuesta_error.status_code == 400
assert "error" in cuerpo_error
print("✅ La API maneja correctamente la solicitud incompleta (error 400, no un fallo interno).")

In [ ]:
import os
os.makedirs("tests", exist_ok=True)

In [ ]:
%%writefile tests/test_preprocesamiento_api.py
import sys, os
import numpy as np
import pytest

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from preprocesamiento_api import extraer_caracteristicas


def test_extraer_caracteristicas_orden_correcto():
    entrada = {"temp_max": 20.0, "temp_min": 5.0, "viento_max": 10.0}
    resultado = extraer_caracteristicas(entrada)
    assert resultado.shape == (1, 3)
    assert np.allclose(resultado, [[20.0, 5.0, 10.0]])


def test_extraer_caracteristicas_campo_faltante():
    entrada = {"temp_max": 20.0, "viento_max": 10.0}
    with pytest.raises(ValueError):
        extraer_caracteristicas(entrada)

In [ ]:
!python -m pytest tests/ -v